# HR 웹 데모 - 설정 및 실행

이 노트북은 HR 직원 관리 Gateway 웹 인터페이스를 시작하는 데 사용됩니다.

## 구성 요소
- **Backend**
- **Frontend**
- **인증**
- **기능** - 직원 검색, 휴가 관리, PII 마스킹 시연

## 1. 설정 및 임포트

In [ ]:
import sys
import json
import os
import subprocess
import time
import socket
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
os.chdir(NOTEBOOK_DIR)

notebook_root = NOTEBOOK_DIR
hr_demo_path = notebook_root / 'hr-web-demo'
pii_gateway_path = notebook_root / 'pii-masking-gateway'

print(f"✓ Notebook root: {notebook_root}")
print(f"✓ HR Demo path: {hr_demo_path}")
print(f"✓ PII Gateway path: {pii_gateway_path}")

# 필요한 디렉토리 확인
if not hr_demo_path.exists():
    print(f"\n❌ HR Demo 디렉토리를 찾을 수 없습니다: {hr_demo_path}")
    print("bedrock-agentcore-gateway 디렉토리에서 이 노트북을 실행해주세요.")
else:
    print(f"\n✓ HR Demo 디렉토리 확인 완료")

## 2. 환경 검증 (Gateway, Python, Node.js)

In [ ]:
# --- Gateway 배포 확인 ---
deployment_file = pii_gateway_path / 'deployment_info.json'

if not deployment_file.exists():
    print("❌ Deployment info를 찾을 수 없습니다!")
    print(f"예상 위치: {deployment_file}")
    print("\n먼저 01-pii-gateway-setup.ipynb를 실행하여 Gateway를 배포해주세요.")
else:
    with open(deployment_file, 'r') as f:
        deployment_info = json.load(f)
    
    print("✅ Gateway 배포 정보 확인 완료")
    print(f"   Gateway URL: {deployment_info['gateway_url']}")
    print(f"   Deployment ID: {deployment_info['deployment_id']}")
    print(f"   Region: {deployment_info['region']}")
    print(f"\n   사용 가능한 클라이언트:")
    for role, info in deployment_info['clients'].items():
        print(f"     - {role}: {info['description']}")

# --- 현재 Python 환경 확인 ---
print(f"\n✅ Python: {sys.executable}")

# --- SageMaker 환경 감지 (Node.js 체크 전에 PATH 설정 필요) ---
IS_SAGEMAKER = False
hostname = os.environ.get('HOSTNAME', '')
if 'sagemaker' in hostname or Path('/home/ec2-user/SageMaker').exists():
    IS_SAGEMAKER = True
    # SageMaker 노트북 커널에서 subprocess로 node/npm 실행 시 PATH에 JupyterSystemEnv가 빠져있음
    jupyter_bin = '/home/ec2-user/anaconda3/envs/JupyterSystemEnv/bin'
    if jupyter_bin not in os.environ['PATH']:
        os.environ['PATH'] = jupyter_bin + ':' + os.environ['PATH']
    print(f"\n✅ SageMaker environment detected")
    print(f"   PATH updated: {jupyter_bin}")
else:
    print("\nℹ️  Local environment (non-SageMaker)")

# --- Node.js 확인 (SageMaker PATH 설정 이후) ---
if shutil.which('node'):
    node_ver = subprocess.run(['node', '--version'], capture_output=True, text=True)
    print(f"✅ Node.js: {node_ver.stdout.strip()}")
else:
    print("\n❌ Node.js를 찾을 수 없습니다!")
    print("   Node.js 16+ 설치: https://nodejs.org/")

## 3. 데모 사용자 계정

데모에는 서로 다른 역할을 가진 3명의 사용자가 사전 구성되어 있습니다:

In [ ]:
print("사용 가능한 데모 사용자:")
print("="*80)

print("\n1. HR Manager - 전체 접근 권한")
print("   사용자명: hr-manager")
print("   비밀번호: hr123!")
print("   Employee ID: EMP-001 / Department: Human Resources")
print("   - 모든 직원의 전체 PII 조회 가능")
print("   - 모든 휴가 기록 관리 가능")
print("   - PII 마스킹 적용 안됨")

print("\n2. Employee 2 - Engineering (Senior Developer)")
print("   사용자명: employee2")
print("   비밀번호: emp789!")
print("   Employee ID: EMP-002 / Department: Engineering")
print("   - 본인 정보 조회 (마스킹 없음)")
print("   - 동료 검색 (PII 마스킹 적용)")
print("   - 본인 휴가 기록 조회")

print("\n3. Employee 3 - Marketing (Marketing Specialist)")
print("   사용자명: employee3")
print("   비밀번호: marketing2024!")
print("   Employee ID: EMP-003 / Department: Marketing")
print("   - 본인 정보 조회 (마스킹 없음)")
print("   - 동료 검색 (PII 마스킹 적용)")
print("   - 본인 휴가 기록 조회")

print("\n" + "="*80)

## 4. 데모 시작 (Backend + Frontend)

이 셀을 실행하면 백엔드와 프론트엔드 서버를 백그라운드로 시작합니다.

**참고**: 서버가 백그라운드에서 실행되므로 다른 셀을 계속 실행할 수 있습니다.

In [ ]:
def check_port(port):
    """포트가 사용 중인지 확인"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

def wait_for_port(port, timeout=15):
    """포트가 열릴 때까지 최대 timeout초 대기"""
    for _ in range(timeout):
        if check_port(port):
            return True
        time.sleep(1)
    return False

python_bin = sys.executable

print("HR Web Demo 시작 중...")
print("="*80)

# 이미 실행 중인지 확인
backend_running = check_port(8000)
frontend_running = check_port(5173)

if backend_running and frontend_running:
    print("\n✅ 서버가 이미 실행 중입니다!")
    print("   • Backend: http://localhost:8000")
    print("   • Frontend: http://localhost:5173")
    print("\n서버를 재시작하려면 먼저 섹션 5에서 서버를 중지하세요.")
else:
    # ========== Backend 시작 ==========
    if not backend_running:
        print("\n📡 Backend 서버 시작 중...")
        backend_path = hr_demo_path / 'backend'
        os.chdir(backend_path)
        
        # 의존성 빠른 체크: 이미 설치되어 있으면 pip install 건너뛰기
        dep_check = subprocess.run(
            [python_bin, '-c', 'import fastapi, uvicorn, strands, sse_starlette'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        
        if dep_check.returncode == 0:
            print("   ✅ Dependencies already installed, skipping...")
        else:
            print("   Installing dependencies...")
            subprocess.run([python_bin, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
        
        print("   Starting uvicorn...")
        cmd = f"{python_bin} -m uvicorn main:app --reload --host 0.0.0.0 --port 8000 > ../backend.log 2>&1 &"
        subprocess.Popen(cmd, shell=True, executable='/bin/bash')
        
        print("   Waiting for backend to be ready...")
        if wait_for_port(8000, timeout=15):
            print("   ✅ Backend 시작 완료: http://localhost:8000")
        else:
            print("   ❌ Backend 시작 실패. backend.log를 확인하세요.")
    else:
        print("\n✅ Backend가 이미 실행 중입니다")
    
    # ========== Frontend 시작 ==========
    if not frontend_running:
        print("\n🎨 Frontend 서버 시작 중...")
        frontend_path = hr_demo_path / 'frontend'
        os.chdir(frontend_path)
        
        # node_modules 확인 및 복구
        node_modules = frontend_path / 'node_modules'
        vite_bin = node_modules / '.bin' / 'vite'
        vite_cli = node_modules / 'vite' / 'dist' / 'node' / 'cli.js'
        
        needs_reinstall = (
            not node_modules.exists() or
            not vite_bin.exists() or
            not vite_cli.exists()
        )
        
        if needs_reinstall:
            print("   📦 Cleaning corrupted dependencies...")
            subprocess.run(['rm', '-rf', 'node_modules', 'package-lock.json'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("   Installing fresh dependencies (this may take a minute)...")
            result = subprocess.run(['npm', 'install'], capture_output=True, text=True)
            if result.returncode != 0:
                print(f"   ❌ 의존성 설치 실패: {result.stderr[:200]}")
            else:
                print("   ✅ Dependencies installed successfully")
        else:
            # 의존성 최신 상태 확인
            subprocess.run(['npm', 'install'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        if IS_SAGEMAKER:
            # SageMaker: Build and serve with Python
            print("   Building frontend for SageMaker...")
            subprocess.run(['rm', '-rf', 'dist', 'node_modules/.vite'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            env = os.environ.copy()
            env['VITE_BASE_PATH'] = '/proxy/5173/'
            env['VITE_API_BASE_URL'] = '/proxy/8000'
            build_result = subprocess.run(['npm', 'run', 'build'],
                                        capture_output=True, text=True, env=env)
            if build_result.returncode != 0:
                print(f"   ❌ Frontend 빌드 실패: {build_result.stderr[:200]}")
            else:
                print("   Starting Python HTTP server...")
                dist_path = frontend_path / 'dist'
                os.chdir(dist_path)
                cmd = f"{python_bin} -m http.server 5173 --bind 0.0.0.0 > ../../frontend.log 2>&1 &"
                subprocess.Popen(cmd, shell=True, executable='/bin/bash')
        else:
            # Local: Use Vite dev server
            print("   Starting Vite dev server...")
            cmd = "npm run dev > ../frontend.log 2>&1 &"
            subprocess.Popen(cmd, shell=True, executable='/bin/bash')
        
        print("   Waiting for frontend to be ready...")
        if wait_for_port(5173, timeout=15):
            print("   ✅ Frontend 시작 완료: http://localhost:5173")
        else:
            print("   ❌ Frontend 시작 실패. frontend.log를 확인하세요.")
    else:
        print("\n✅ Frontend가 이미 실행 중입니다")
    
    print("\n" + "="*80)
    print("\n🎉 데모 서버가 백그라운드에서 실행 중입니다!")
    
    if IS_SAGEMAKER:
        import json as _json
        with open('/opt/ml/metadata/resource-metadata.json', 'r') as _f:
            _nb_name = _json.load(_f)['ResourceName']
        _nb_region = boto3.session.Session().region_name if 'boto3' in dir() else REGION if 'REGION' in dir() else 'us-east-1'
        _nb_url = f'https://{_nb_name}.notebook.{_nb_region}.sagemaker.aws'
        print(f"\n📋 SageMaker 접속 정보:")
        print(f"   • Frontend: {_nb_url}/proxy/5173/")
        print(f"   • Backend API: {_nb_url}/proxy/8000/")
    else:
        print("\n📋 접속 정보:")
        print("   • Frontend (Web UI): http://localhost:5173")
        print("   • Backend API: http://localhost:8000")
        print("   • API 문서: http://localhost:8000/docs")
    
    print("\n📝 Logs:")
    print("   • Backend logs: hr-web-demo/backend.log")
    print("   • Frontend logs: hr-web-demo/frontend.log")
    print("\n🛑 서버 중지: 섹션 5의 '서버 중지' 셀을 실행하세요.")

os.chdir(notebook_root)

## 5. 서버 중지

아래 셀을 실행하면 Backend(8000)와 Frontend(5173) 서버를 모두 중지합니다.

In [ ]:
import subprocess

print("🛑 데모 서버 중지 중...")
print("="*80)

for port, name in [(8000, 'Backend'), (5173, 'Frontend')]:
    result = subprocess.run(
        f"lsof -ti :{port}",
        shell=True, capture_output=True, text=True
    )
    pids = result.stdout.strip()
    if pids:
        subprocess.run(f"lsof -ti :{port} | xargs kill", shell=True)
        print(f"   ✅ {name} (port {port}) 중지 완료 (PID: {pids.replace(chr(10), ', ')})")
    else:
        print(f"   ℹ️  {name} (port {port}) 실행 중이 아닙니다")

print("\n" + "="*80)
print("\n서버를 다시 시작하려면 섹션 4를 다시 실행하세요.")

## 요약

이 노트북에서 수행하는 작업:

1. ✓ Gateway 배포, Python 환경, Node.js 검증
2. ✓ SageMaker 환경 자동 감지
3. ✓ 백엔드 및 프론트엔드 서버 백그라운드 실행
4. ✓ 데모 사용자 계정 확인
5. ✓ 단일 셀로 서버 중지

### 주요 기능

- **이중 인터셉터 아키텍처**: Request + Response 인터셉터 패턴
- **세분화된 접근 제어**: 역할 기반 도구 필터링
- **PII 마스킹**: Bedrock Guardrails를 활용한 AI 기반 탐지
- **Self-Access 감지**: 직원이 본인 데이터를 조회할 때 마스킹 미적용
- **다중 역할 지원**: HR Manager, Employee 역할별 차등 권한